<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-05-bigquery-ml/lesson-5.3-llm-in-sql/notebooks/GCP_Capstone_5.3_LLM_SQL.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5.3 LLM in SQL — AI.GENERATE, VECTOR_SEARCH & Embeddings
**Netsetos GenAI Engineering — GCP Capstone**

Call Gemini from SQL. Generate embeddings. Run vector search. Build RAG — all without Python.


## Setup


In [ ]:
from google.colab import auth
auth.authenticate_user()

PROJECT_ID = 'documind-ai-YOUR-ID'  # CHANGE

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)

# Ensure this module's datasets exist (idempotent -- BigQuery never auto-creates them).
# NOTE: this notebook is self-contained -- the fixture cell below seeds
# rag_data.doc_chunks from an inline list, so no Lesson 5.1 table is required.
for _ds in ('rag_data', 'ml_models'):
    _d = bigquery.Dataset(f'{PROJECT_ID}.{_ds}'); _d.location = 'US'
    client.create_dataset(_d, exists_ok=True)

def run_query(sql):
    return client.query(sql).to_dataframe()

def run_ddl(sql):
    job = client.query(sql)
    job.result()
    print(f'Done: {job.num_dml_affected_rows or "OK"}')

print(f'Connected to {PROJECT_ID}')


## Cell 1: Create Remote Models


In [ ]:
# Create connection + remote models
# gemini_flash is registered so you can see the remote-model pattern (and because
# the LEGACY ML.GENERATE_TEXT TVF requires one). This lesson calls AI.GENERATE,
# which is scalar and takes an endpoint directly -- it never reads this model.
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.gemini_flash`
  REMOTE WITH CONNECTION DEFAULT
  OPTIONS (ENDPOINT = 'gemini-3.6-flash')
''')
print('Gemini model created (reference only -- AI.GENERATE does not use it)')

# THE embedding model for the DocuMind corpus. Lesson 2.4 settled it:
# gemini-embedding-001, with output_dimensionality pinned to 768 at every call.
# One model per store -- never mix embedding models inside one vector table.
run_ddl(f'''
CREATE OR REPLACE MODEL `{PROJECT_ID}.ml_models.embed_gemini_768`
  REMOTE WITH CONNECTION DEFAULT
  OPTIONS (ENDPOINT = 'gemini-embedding-001')
''')
print('Embedding model created (gemini-embedding-001, pin 768 dims per call)')


## Fixture: doc_chunks — the table the lesson queries

**Self-seeding.** The cell below builds `rag_data.doc_chunks` from an inline VALUES list of
12 synthetic ACME policy clauses — the same corpus Lesson 4.5 uses — so this notebook runs
on a fresh project with **no dependency on Lesson 5.1**. `chunk_text` is real prose (two to
three sentences per row), because AI.GENERATE summaries and the RAG answer need something
longer than a document title to work with.


In [ ]:
# Fixture: rag_data.doc_chunks -- the table this lesson teaches.
# 12 synthetic ACME policy clauses (no real PII), same corpus as Lesson 4.5.
# Prose chunk_text, so the summaries, the classifier and the RAG answer below
# have real sentences to work on. Columns: chunk_id, doc_id, chunk_text, doc_type.
ACME_CHUNKS = [
    (1, 'acme_hr_handbook', 'LV-01 Leave Policy. Earned leave (EL) accrues at 1.5 days per month of service. Unused EL carries forward up to a maximum balance of 30 days; balance above 30 days lapses on 31 March. Leave without pay (LWP) beyond 5 days requires HRBP approval.', 'hr_policy'),
    (2, 'acme_hr_handbook', 'LV-07 Leave Encashment. Confirmed employees may encash up to 15 days of earned leave once per financial year. Encashment is paid at basic pay with the March payroll and is subject to TDS. Employees serving notice or on probation are not eligible for encashment.', 'hr_policy'),
    (3, 'acme_hr_handbook', 'NP-03 Notice Period. The notice period is 60 days for grades E1 to E3 and 90 days for grade M1 and above. Notice may be bought out at basic pay with business-head approval. Unused earned leave cannot be adjusted against the notice period.', 'hr_policy'),
    (4, 'acme_hr_handbook', 'PB-02 Probation. Probation lasts 6 months from the date of joining and ends with a confirmation review. During probation the notice period is 15 days for both sides. Probationers accrue earned leave but may not encash it.', 'hr_policy'),
    (5, 'acme_finance_policy', 'EXP-12 Travel Expenses. Domestic travel per diem is Rs 2,500 in metro cities and Rs 1,800 elsewhere. The reimbursement cap is Rs 40,000 per trip. Claims are filed in Concur within 30 days with a GST invoice for any single bill above Rs 5,000.', 'finance_policy'),
    (6, 'acme_finance_policy', 'EXP-15 Client Entertainment. Client entertainment is capped at Rs 6,000 per event and Rs 25,000 per client per quarter. Spend above the cap needs VP approval before the event. Alcohol is reimbursable only for client-facing events.', 'finance_policy'),
    (7, 'acme_finance_policy', 'PR-05 Payroll and Tax. Investment proofs are due by 31 January; Form 16 is issued by 15 June. Flexi benefits (meal card, fuel, telephone) are elected once a year in April. Salary is credited on the last working day of the month.', 'finance_policy'),
    (8, 'acme_hr_handbook', 'WFH-01 Hybrid Work. Employees work 3 days from office and up to 2 days remote per week. A one-time ergonomic allowance of Rs 15,000 is reimbursable against invoices. Fully remote work needs VP approval and is reviewed every 6 months.', 'hr_policy'),
    (9, 'acme_it_policy', 'IT-SEC-04 Device Security. All laptops use full-disk encryption and MFA. Passwords rotate every 90 days. USB mass storage is blocked; exceptions go through the IT service desk with manager approval.', 'it_policy'),
    (10, 'acme_it_policy', 'AI-02 AI Tool Usage. Customer PII must never be pasted into public LLM tools. Approved tools are DocuMind and Vertex AI inside the ACME project. Generated content used externally must be reviewed by a human owner.', 'it_policy'),
    (11, 'acme_hr_handbook', 'POSH-01 POSH Committee. Complaints may be filed with the Internal Committee within 3 months of the incident. The inquiry is completed within 90 days and the report is shared with both parties within 10 days.', 'hr_policy'),
    (12, 'acme_legal_policy', 'VC-09 Vendor Contracts. Vendor engagements need an MSA plus a signed SOW. Standard payment terms are net 45 days. Liability is capped at 12 months of fees unless Legal approves an exception.', 'legal_policy'),
]
assert not any("'" in c[2] for c in ACME_CHUNKS), 'no apostrophes: they would break the SQL literals'

_head = ("STRUCT({} AS chunk_id, '{}' AS doc_id, '{}' AS chunk_text, '{}' AS doc_type)"
         .format(*ACME_CHUNKS[0]))
_tail = ["({}, '{}', '{}', '{}')".format(*r) for r in ACME_CHUNKS[1:]]
_values = ',\n  '.join([_head] + _tail)

run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.doc_chunks` AS
SELECT * FROM UNNEST([
  {_values}
])
''')

# Which path did the fixture take? (Lesson 5.1's table is no longer required.)
_legacy = list(client.query(
    f"SELECT table_name FROM `{PROJECT_ID}.rag_data.INFORMATION_SCHEMA.TABLES` "
    "WHERE table_name = 'document_features'").result())
print(f'doc_chunks seeded INLINE from {len(ACME_CHUNKS)} ACME policy chunks.')
print("Lesson 5.1's rag_data.document_features: " + (
    'present in this project, deliberately NOT used -- its title column is 2-3 words, too short to summarize.'
    if _legacy else 'absent -- fine, this notebook does not depend on it.'))
print(run_query(f'SELECT chunk_id, doc_id, doc_type, LEFT(chunk_text, 60) AS preview FROM `{PROJECT_ID}.rag_data.doc_chunks` ORDER BY chunk_id LIMIT 3'))


## Cell 2: AI.GENERATE — Summarize Documents


In [ ]:
# Summarize document chunks with AI.GENERATE (Gemini runs per row, in SQL).
# endpoint => 'gemini-3.6-flash' pins the model; without it AI.GENERATE uses
# BigQuery's default endpoint. AI.GENERATE is SCALAR: it takes an endpoint
# directly and never reads the gemini_flash remote model registered above.
results = run_query(rf'''
SELECT
  chunk_id, chunk_text,
  AI.GENERATE(
    CONCAT('Summarize in one sentence:\n', chunk_text),
    endpoint => 'gemini-3.6-flash'
  ).result AS summary
FROM `{PROJECT_ID}.rag_data.doc_chunks`
LIMIT 10
''')
print(results)


## Cell 3: ML.GENERATE_EMBEDDING — Text to Vectors


In [ ]:
# Generate embeddings for the chunks (RETRIEVAL_DOCUMENT task type).
# gemini-embedding-001 returns 3072 dims by DEFAULT. Pin 768 or the vectors stop
# matching the 768-dim contract the rest of the course (and Lesson 2.4) uses.
run_ddl(f'''
CREATE OR REPLACE TABLE `{PROJECT_ID}.rag_data.chunk_embeddings` AS
SELECT
  chunk_id, doc_id, chunk_text, doc_type,
  ml_generate_embedding_result AS embedding,
  ml_generate_embedding_status AS status
FROM ML.GENERATE_EMBEDDING(
  MODEL `{PROJECT_ID}.ml_models.embed_gemini_768`,
  (SELECT chunk_id, doc_id, chunk_text, doc_type, chunk_text AS content
   FROM `{PROJECT_ID}.rag_data.doc_chunks`),
  STRUCT(TRUE AS flatten_json_output,
         'RETRIEVAL_DOCUMENT' AS task_type,
         768 AS output_dimensionality)
)
WHERE ml_generate_embedding_status = ''
''')
print('Embeddings generated')

# Verify dimensions -- must be 768, not 3072
result = run_query(f'''
SELECT chunk_id, ARRAY_LENGTH(embedding) AS dims
FROM `{PROJECT_ID}.rag_data.chunk_embeddings`
LIMIT 5
''')
print(result)


## Cell 4: VECTOR_SEARCH — Find Similar Documents


In [ ]:
# Search for chunks similar to a query.
# The query embedding MUST use the same model and the same 768 dims as the
# stored vectors -- only task_type changes (RETRIEVAL_QUERY, not _DOCUMENT).
results = run_query(f'''
SELECT
  base.chunk_id, base.chunk_text, distance
FROM VECTOR_SEARCH(
  TABLE `{PROJECT_ID}.rag_data.chunk_embeddings`,
  'embedding',
  (SELECT ml_generate_embedding_result AS embedding
   FROM ML.GENERATE_EMBEDDING(
     MODEL `{PROJECT_ID}.ml_models.embed_gemini_768`,
     (SELECT 'How much notice do I have to serve before resigning?' AS content),
     STRUCT('RETRIEVAL_QUERY' AS task_type, 768 AS output_dimensionality))),
  top_k => 5,
  distance_type => 'COSINE'
)
ORDER BY distance
''')
print('=== Vector Search Results ===')
print(results)


## Cell 4b: Vector index — and why coverage stays 0% on this fixture

BigQuery does not populate a vector index until the base table is **at least 10 MB**
(index-unused reason `BASE_TABLE_TOO_SMALL`). Twelve chunks are nowhere near that, so the
index below is created and then simply never built — `coverage_percentage` stays 0 and
every `VECTOR_SEARCH` above silently runs a brute-force scan. That is correct behaviour,
and it is why you cannot demo an index speed-up on a toy table.


In [ ]:
# Create the IVF index, then look at what BigQuery actually did with it.
# Expect coverage_percentage = 0 here: the base table is far under 10 MB.
try:
    run_ddl(f'''
    CREATE OR REPLACE VECTOR INDEX documind_chunk_idx
    ON `{PROJECT_ID}.rag_data.chunk_embeddings`(embedding)
    OPTIONS (index_type = 'IVF',
             distance_type = 'COSINE',
             ivf_options = '{{"num_lists": 10}}')
    ''')
    print('Vector index submitted')
except Exception as e:
    print(f'CREATE VECTOR INDEX: {e}')

print(run_query(f'''
SELECT table_name, index_name, index_status, coverage_percentage
FROM `{PROJECT_ID}.rag_data.INFORMATION_SCHEMA.VECTOR_INDEXES`
'''))
print('coverage_percentage 0 / index never ACTIVE = BASE_TABLE_TOO_SMALL (< 10 MB).')
print('VECTOR_SEARCH still works -- it just runs brute force. Do not chase a timing win here.')


## Cell 5: RAG-in-SQL — Complete Pipeline


In [ ]:
# Complete RAG in ONE statement: embed query -> VECTOR_SEARCH -> AI.GENERATE.
# AI.GENERATE is SCALAR, so the retrieval half becomes a subquery that yields a
# single prompt row and .result is the answer column. No temperature / top_p /
# top_k anywhere: gemini-3.6-flash ignores them. Tune length instead.
result = run_query(rf'''
SELECT
  AI.GENERATE(
    prompt,
    endpoint => 'gemini-3.6-flash',
    model_params => JSON '{{"generation_config":{{"max_output_tokens":1024}}}}'
  ).result AS answer
FROM (
  SELECT CONCAT(
     'Answer using ONLY this context:\n',
     STRING_AGG(
       FORMAT('[Source %d] %s', base.chunk_id, base.chunk_text),
       '\n'),
     '\n\nQuestion: ', MAX(query.q)
   ) AS prompt
   FROM VECTOR_SEARCH(
     TABLE `{PROJECT_ID}.rag_data.chunk_embeddings`,
     'embedding',
     (SELECT ml_generate_embedding_result AS embedding, content AS q
      FROM ML.GENERATE_EMBEDDING(
        MODEL `{PROJECT_ID}.ml_models.embed_gemini_768`,
        (SELECT 'How many days of earned leave can I encash in a year?' AS content),
        STRUCT('RETRIEVAL_QUERY' AS task_type, 768 AS output_dimensionality))),
     top_k => 3,
     distance_type => 'COSINE'))
''')
print('=== RAG Answer ===')
print(result.iloc[0]['answer'] if len(result) > 0 else 'No result')


## Cell 6: Stored Procedure — ask_documind()


In [ ]:
# Create the ask_documind stored procedure -- AI.GENERATE, not the legacy TVF.
run_ddl(rf'''
CREATE OR REPLACE PROCEDURE `{PROJECT_ID}.ml_models.ask_documind`(
  user_question STRING)
BEGIN
  SELECT AI.GENERATE(
           prompt,
           endpoint => 'gemini-3.6-flash',
           model_params => JSON '{{"generation_config":{{"max_output_tokens":1024}}}}'
         ).result AS answer
  FROM (
    SELECT CONCAT(
       'Answer from context only:\n',
       STRING_AGG(FORMAT('[%d] %s', base.chunk_id, base.chunk_text), '\n'),
       '\n\nQ: ', MAX(query.q)
     ) AS prompt
     FROM VECTOR_SEARCH(
       TABLE `{PROJECT_ID}.rag_data.chunk_embeddings`, 'embedding',
       (SELECT ml_generate_embedding_result AS embedding, content AS q
        FROM ML.GENERATE_EMBEDDING(
          MODEL `{PROJECT_ID}.ml_models.embed_gemini_768`,
          (SELECT user_question AS content),
          STRUCT('RETRIEVAL_QUERY' AS task_type, 768 AS output_dimensionality))),
       top_k => 3, distance_type => 'COSINE'));
END
''')
print('ask_documind() procedure created')

# Test it
try:
    result = run_query(
        f"CALL `{PROJECT_ID}.ml_models.ask_documind`('What is the notice period for grade M1?')")
    print(result)
except Exception as e:
    print(f'Note: {e}')


## Cell 7: Bulk Operations — Classify + Extract


In [ ]:
# Zero-shot classification with AI.GENERATE (endpoint pins gemini-3.6-flash).
# Labels match the doc_type vocabulary the fixture seeds, so predicted vs actual
# is a real comparison rather than a guess against unrelated categories.
results = run_query(rf'''
SELECT
  chunk_id, chunk_text,
  AI.GENERATE(
    CONCAT('Classify this ACME policy clause as exactly one of: hr_policy, finance_policy, it_policy, legal_policy.\n\nText: ', chunk_text),
    endpoint => 'gemini-3.6-flash'
  ).result AS predicted_type,
  doc_type AS actual_type
FROM `{PROJECT_ID}.rag_data.doc_chunks`
''')
print('=== LLM Classification vs Actual ===')
print(results[['chunk_id','chunk_text','predicted_type','actual_type']])


## ✅ Lesson 5.3 Complete!

**Three AI functions mastered:**
- ✅ AI.GENERATE — Gemini in every row (summarize, classify, extract, translate). The legacy
  ML.GENERATE_TEXT TVF is not called anywhere in this lesson — it survives only as a row in
  the comparison table.
- ✅ ML.GENERATE_EMBEDDING — text to 768-dim vectors in SQL with gemini-embedding-001,
  `output_dimensionality` pinned (it returns 3072 by default). One embedding model per store.
- ✅ VECTOR_SEARCH — nearest neighbours with COSINE distance, plus why an IVF index sits at
  0% coverage until the base table passes 10 MB.
- ✅ RAG-in-SQL — the whole pipeline in one statement
- ✅ ask_documind() stored procedure

**Module 5 so far:**
- 5.1: CREATE MODEL (LINEAR_REG, LOGISTIC_REG, KMEANS)
- 5.2: ARIMA_PLUS (forecast, anomaly, decompose)
- 5.3: AI functions (AI.GENERATE, VECTOR_SEARCH, embeddings) — you are here

**Next: Lesson 5.4 — BigQuery to Vertex AI: Model Registry, DataFrames & Feature Engineering.**
After that, Lesson 5.5 — Engineer Retrieval Features.
